# Expert-CFG 베이스라인 재현 — VQA-RAD

논문: *Uncertainty-Driven Expert Control: Enhancing the Reliability of Medical Vision-Language Models* (arXiv:2507.09209)

**재현 목표 (Table 1, Phi3.5V-Med 4.2B, VQA-RAD)**

| Open (recall) | Closed (acc) | Overall |
|---|---|---|
| 69.27 | 90.44 | 82.04 |

VQA-RAD: test 451 (open 179 / closed 272) — PathVQA 보다 훨씬 작아 빠른 검증에 적합합니다.

In [1]:
!git clone https://github.com/Minseongkim-dev/Expert-CFG.git /content/Expert-CFG 2>/dev/null || \
    (cd /content/Expert-CFG && git pull)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR  = '/content/drive/MyDrive/expert_cfg/logs_phi35_vqarad_baseline'
RESULT_JSON = '/content/drive/MyDrive/expert_cfg/vqarad_baseline_test.json'

Mounted at /content/drive


In [3]:
# peft 0.21+ 는 torchao>=0.16.0 을 요구합니다
import importlib, subprocess, sys

def _pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

_torchao_ver = _pkg_version('torchao')
if _torchao_ver is None or tuple(int(x) for x in _torchao_ver.split('.')[:2]) < (0, 16):
    print(f'torchao {_torchao_ver} → 업그레이드 중...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'])
    print('완료 (런타임 재시작 불필요)')
else:
    print(f'torchao {_torchao_ver} OK')

torchao 0.10.0 → 업그레이드 중...
완료 (런타임 재시작 불필요)


In [4]:
import os, sys, json, glob, random
from pathlib import Path

import numpy as np
import torch

def find_repo_root():
    for start in [Path("/content/Expert-CFG"), Path.cwd()]:
        if start.exists() and (start / "src" / "highlighter_modules").is_dir():
            return start.resolve()
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "src" / "highlighter_modules").is_dir():
            return parent
    raise FileNotFoundError("repo root 를 찾지 못했습니다. /content/Expert-CFG 에 clone 하세요.")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import transformers, peft

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

print("repo         :", REPO_ROOT)
print("python       :", sys.version.split()[0])
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("BF16         :", USE_BF16)

repo         : /content/Expert-CFG
python       : 3.13.16
torch        : 2.11.0+cu130
transformers : 5.20.2
peft         : 0.21.2
GPU          : Tesla T4
BF16         : True


In [5]:
# ── 경로 ────────────────────────────────────────────────────────────────
BASE_MODEL_DIR = './Phi-3.5-vision-instruct'
MED_LORA_DIR   = './lora_weights/logs_phi35_pubmed_instruct'
MED_LORA_HF    = 'ecoxial2007/Phi-3.5V-Med'
VQARAD_DIR     = './data/VQA_RAD'

# ── LoRA (논문 §4.3) ─────────────────────────────────────────────────────
LORA_RANK        = 64
LORA_ALPHA_RATIO = 2
LORA_DROPOUT     = 0.0
LORA_TARGET      = ['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj']
FREEZE_VISION    = True

# ── 학습 (§4.3) ──────────────────────────────────────────────────────────
LR            = 1e-4
GLOBAL_BATCH  = 64
EPOCHS        = 5
PER_DEVICE_BS = 1
WORLD_SIZE    = int(os.environ.get('WORLD_SIZE', 1))
GRAD_ACCUM    = GLOBAL_BATCH // (PER_DEVICE_BS * WORLD_SIZE)
WARMUP_RATIO  = 0.03
WEIGHT_DECAY  = 0.0

# ── 추론 ─────────────────────────────────────────────────────────────────
NUM_CROPS      = 16
MAX_NEW_TOKENS = 64
NUM_BEAMS      = 1
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULT_JSON), exist_ok=True)
print(f'grad_accum={GRAD_ACCUM}  (global batch {GLOBAL_BATCH})')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

grad_accum=64  (global batch 64)
OUTPUT_DIR = /content/drive/MyDrive/expert_cfg/logs_phi35_vqarad_baseline


In [8]:
from PIL import Image
from datasets import load_dataset

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

# VQA-RAD HuggingFace 후보 (순서대로 시도)
_HF_TRAIN_CANDIDATES = [
    ('flavell/vqa_rad',                  'train'),
    ('BioMedVLP/VQA-RAD',               'train'),
    ('AIR-Bench/qa_vqa_rad',             'train'),
]
_HF_TEST_CANDIDATES = [
    ('flavell/vqa_rad',                  'test'),
    ('BioMedVLP/VQA-RAD',               'test'),
    ('Multimodal-Fatima/VQA_RAD_test',   'test'),
    ('AIR-Bench/qa_vqa_rad',             'test'),
]

def _try_load_hf(candidates, cache_dir):
    for ds_id, split in candidates:
        try:
            ds = load_dataset(ds_id, split=split, cache_dir=cache_dir)
            print(f'  로드 성공: {ds_id} / {split}  ({len(ds)} samples)')
            return ds, ds_id
        except Exception as e:
            print(f'  {ds_id}/{split} 실패: {type(e).__name__}')
    raise RuntimeError(
        'VQA-RAD 데이터를 자동으로 가져오지 못했습니다.\n'
        '수동 다운로드 방법:\n'
        '  1) https://osf.io/89kps/ 에서 다운로드 후\n'
        f'     {VQARAD_DIR}/train.json, test.json 과\n'
        f'     {VQARAD_DIR}/images/ 폴더에 배치\n'
        '  2) 또는 HF 토큰을 설정하세요: from huggingface_hub import login; login()'
    )

# HF 열 이름 정규화 (데이터셋마다 다를 수 있음)
_COL_MAP = {
    'q': 'question', 'a': 'answer', 'img': 'image',
    'question_type': None,  # 무시
}
def _normalize(r):
    q = r.get('question') or r.get('q') or r.get('Question', '')
    a = r.get('answer')   or r.get('a') or r.get('Answer', '')
    img = r.get('image')  or r.get('img')
    return str(q).strip(), str(a).strip(), img

class VQARADSplit(torch.utils.data.Dataset):
    def __init__(self, split):
        local_json = os.path.join(VQARAD_DIR, f'{split}.json')
        if os.path.exists(local_json):
            with open(local_json) as f:
                self.records = json.load(f)
            self.source = 'local'
            self.img_root = os.path.join(VQARAD_DIR, 'images')
            self.ds_id = 'local'
        else:
            print(f'[{split}] HuggingFace 로드 시도...')
            candidates = _HF_TRAIN_CANDIDATES if split == 'train' else _HF_TEST_CANDIDATES
            hf_ds, self.ds_id = _try_load_hf(
                candidates, cache_dir=os.path.join(VQARAD_DIR, 'hf_cache'))
            self.records = hf_ds
            self.source = 'hf'

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        if self.source == 'local':
            img_path = os.path.join(self.img_root, r['img_id'])
            image = Image.open(img_path).convert('RGB')
            q, a = str(r['question']).strip(), str(r['answer']).strip()
        else:
            q, a, img = _normalize(r)
            image = img.convert('RGB') if img else Image.new('RGB', (224, 224))
        return {'image': image, 'question': q, 'answer': a, 'answer_type': answer_type_of(a)}

def type_counts(ds):
    if ds.source == 'local':
        answers = [r['answer'] for r in ds.records]
    else:
        try:
            answers = ds.records['answer']
        except Exception:
            answers = [_normalize(r)[1] for r in ds.records]
    closed = sum(answer_type_of(a) == 'CLOSED' for a in answers)
    return closed, len(answers) - closed

train_ds = VQARADSplit('train')
test_ds  = VQARADSplit('test')

for name, ds in [('train', train_ds), ('test', test_ds)]:
    c, o = type_counts(ds)
    print(f'{name:5s} [{ds.ds_id}] total={len(ds):4d}  closed={c:3d}  open={o:3d}')

[train] HuggingFace 로드 시도...
  flavell/vqa_rad/train 실패: DatasetNotFoundError
  BioMedVLP/VQA-RAD/train 실패: DatasetNotFoundError
  AIR-Bench/qa_vqa_rad/train 실패: DatasetNotFoundError


RuntimeError: VQA-RAD 데이터를 자동으로 가져오지 못했습니다.
수동 다운로드 방법:
  1) https://osf.io/89kps/ 에서 다운로드 후
     ./data/VQA_RAD/train.json, test.json 과
     ./data/VQA_RAD/images/ 폴더에 배치
  2) 또는 HF 토큰을 설정하세요: from huggingface_hub import login; login()

In [ ]:
import shutil
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel, LoraConfig, get_peft_model

_SKIP_FILES = {'processor_config.json', 'processing_phi3_v_cfg.py'}

# 1) 기본 가중치 다운로드
if not os.path.exists(os.path.join(BASE_MODEL_DIR, 'config.json')):
    print('Phi-3.5-vision-instruct 가중치 다운로드 중 (~8 GB)...')
    _hf_tmp = snapshot_download('microsoft/Phi-3.5-vision-instruct',
                                local_dir='/tmp/phi35_hf',
                                ignore_patterns=['*.git', '__pycache__'])
    os.makedirs(BASE_MODEL_DIR, exist_ok=True)
    for _f in os.listdir(_hf_tmp):
        if _f in _SKIP_FILES:
            continue
        _s, _d = os.path.join(_hf_tmp, _f), os.path.join(BASE_MODEL_DIR, _f)
        (shutil.copytree if os.path.isdir(_s) else shutil.copy2)(
            _s, _d, **({'dirs_exist_ok': True} if os.path.isdir(_s) else {}))
    print('완료')
else:
    print('기본 가중치 이미 존재')

with open(os.path.join(BASE_MODEL_DIR, 'processor_config.json')) as _f:
    assert 'processing_phi3_v_cfg' in _f.read(), \
        'processor_config.json 이 HF 원본으로 덮어써졌습니다 — repo 버전으로 복원하세요'

processor = AutoProcessor.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# 2) Med LoRA 다운로드/확인
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    assert found, f'{MED_LORA_HF} 에서 adapter_config.json 을 찾지 못했습니다'
    return os.path.dirname(found[0])

med_lora_path = resolve_med_lora()
print('Med LoRA :', med_lora_path)

# 3) Base → Med LoRA merge → 새 downstream LoRA 추가 (fine-tuning 준비)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR, torch_dtype=torch.bfloat16,
    trust_remote_code=True, _attn_implementation='eager')

model = PeftModel.from_pretrained(base, med_lora_path).merge_and_unload()

model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK * LORA_ALPHA_RATIO,
    lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET,
    init_lora_weights=True, task_type='CAUSAL_LM'))

if FREEZE_VISION:
    for n, p in model.named_parameters():
        if 'vision_embed_tokens' in n:
            p.requires_grad_(False)

model.print_trainable_parameters()
model.to(DEVICE)
print('모델 준비 완료 (fine-tuning 준비 상태)')

In [ ]:
from transformers import Trainer, TrainingArguments
import inspect, re
from collections import Counter
from tqdm.auto import tqdm

# ── 평가 함수 ─────────────────────────────────────────────────────────
def normalize(text):
    text = text.lower().replace('<|end|>', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def predictive_entropy(step_scores):
    if not step_scores:
        return 0.0
    logits = torch.stack([s[0] for s in step_scores]).float()
    logp = torch.log_softmax(logits, dim=-1)
    return (-(logp.exp() * logp).sum(-1)).mean().item()

@torch.no_grad()
def evaluate(model, processor, dataset, max_samples=None):
    model.eval()
    end_id = processor.tokenizer.convert_tokens_to_ids('<|end|>')
    records = []
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    for i in tqdm(range(n)):
        ex = dataset[i]
        prompt = processor.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': f"<|image_1|>\n{ex['question']}"}],
            tokenize=False, add_generation_prompt=True)
        inputs = processor(prompt, [ex['image']], return_tensors='pt', qs_highlighted_parts=[])
        del inputs['highlight_attention_mask']
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, do_sample=False,
            eos_token_id=[processor.tokenizer.eos_token_id, end_id],
            output_scores=True, return_dict_in_generate=True,
        )
        gen_ids = out.sequences[:, inputs['input_ids'].size(1):]
        pred = processor.batch_decode(gen_ids, skip_special_tokens=True,
                                      clean_up_tokenization_spaces=False)[0].strip().strip('.')
        score = (closed_accuracy if ex['answer_type'] == 'CLOSED' else open_recall)(pred, ex['answer'])
        records.append({'idx': i, 'question': ex['question'], 'answer': ex['answer'],
                        'answer_type': ex['answer_type'], 'prediction': pred,
                        'score': score, 'entropy': predictive_entropy(out.scores)})
    return records

def summarize(records):
    closed = [r['score'] for r in records if r['answer_type'] == 'CLOSED']
    opened = [r['score'] for r in records if r['answer_type'] == 'OPEN']
    return {'open':    100 * np.mean(opened)  if opened  else float('nan'),
            'closed':  100 * np.mean(closed)  if closed  else float('nan'),
            'overall': 100 * np.mean([r['score'] for r in records]),
            'n_open': len(opened), 'n_closed': len(closed)}

# ── collator ──────────────────────────────────────────────────────────
class VQARADCollator:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, examples):
        assert len(examples) == 1
        ex = examples[0]
        prompt = self.processor.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': f"<|image_1|>\n{ex['question']}"}],
            tokenize=False, add_generation_prompt=True)
        answer = f"{ex['answer']}<|end|>\n<|endoftext|>"
        batch = self.processor(prompt, [ex['image']], return_tensors='pt', qs_highlighted_parts=[])
        del batch['highlight_attention_mask']
        prompt_ids = batch['input_ids']
        answer_ids = self.processor.tokenizer(answer, add_special_tokens=False, return_tensors='pt')['input_ids']
        batch['input_ids'] = torch.cat([prompt_ids, answer_ids], dim=1)
        batch['labels']    = torch.cat([torch.full_like(prompt_ids, -100), answer_ids], dim=1)
        del batch['attention_mask']
        return batch

print('평가/학습 함수 정의 완료')

In [ ]:
# ── VQA-RAD Downstream Fine-tuning ───────────────────────────────────
_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
_total_steps = (len(train_ds) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
_v4 = dict(learning_rate=LR, optim='adamw_torch', lr_scheduler_type='cosine', weight_decay=WEIGHT_DECAY)
if 'warmup_ratio' in _ta_params:
    _v4['warmup_ratio'] = WARMUP_RATIO
elif 'warmup_steps' in _ta_params:
    _v4['warmup_steps'] = int(_total_steps * WARMUP_RATIO)
    print(f'warmup_steps={_v4["warmup_steps"]} ({WARMUP_RATIO*100:.0f}% of {_total_steps})')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    save_only_model=True,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to='none',
    seed=SEED,
    **{k: v for k, v in _v4.items() if k in _ta_params},
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_ds, data_collator=VQARADCollator(processor))

_ckpt_dirs = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')))
_resume = _ckpt_dirs[-1] if _ckpt_dirs else False
print(f'resume_from_checkpoint = {_resume}')

trainer.train(resume_from_checkpoint=_resume)
trainer.save_model(OUTPUT_DIR)
print('학습 완료')

In [ ]:
# ── 테스트셋 평가 ─────────────────────────────────────────────────────
records = evaluate(model, processor, test_ds)
with open(RESULT_JSON, 'w') as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

m = summarize(records)
paper = {'open': 69.27, 'closed': 90.44, 'overall': 82.04}
print(f"{'':8s}{'paper':>8s}{'ours':>8s}{'diff':>8s}")
for k in ('open', 'closed', 'overall'):
    print(f'{k:8s}{paper[k]:8.2f}{m[k]:8.2f}{m[k]-paper[k]:+8.2f}')
print(f"n_open={m['n_open']}, n_closed={m['n_closed']}")
print(f'결과 저장: {RESULT_JSON}')